# Primary Econometric Analysis: The COVID-19 Graduate Penalty

## 1. Declaration of Ambition
**Ambition: Causal.**

This analysis attempts to identify the **treatment effect** of the COVID-19 economic shock on graduate full-time employment (FTE) rates. By using a First-Differences framework, we isolate the pandemic-induced change from time-invariant study-area characteristics (fixed effects).

## 2. Econometric Specification

We estimate the following model:
$$\Delta FTE_i = \beta_0 + \beta_1 (Salary_{18,i}) + \epsilon_i$$

- **Functional Form:** Linear OLS on First-Differenced data.
- **Regressors:** Baseline median salary in 2018 ($Salary_{18}$) in thousands of AUD.
- **Sample:** $n=21$ Australian study areas.
- **Error Structure:** Heteroskedasticity-robust standard errors (HC3) due to small sample size.

**Identification Strategy:** OLS-with-First-Differences. We control for all time-invariant confounders (e.g., inherent field difficulty, long-term industry prestige) by differencing the outcome variable. The key assumption is **Parallel Trends**: in the absence of COVID-19, the employment rates of all study areas would have followed the same trend.

In [ ]:
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

path = Path('../data/clean/final_pandemic_research_data.csv')
df = pd.read_csv(path)

df = df[~df['Study_Area'].str.contains('All|Standard deviation', na=False)].copy()
df['FTE_18'] = pd.to_numeric(df['FTE_18'].astype(str).str.replace('%',''), errors='coerce')
df['FTE_20'] = pd.to_numeric(df['FTE_20'].astype(str).str.replace('%',''), errors='coerce')
df['Salary_18_k'] = pd.to_numeric(df['Salary_18'].astype(str).str.replace(',',''), errors='coerce') / 1000
df['delta_fte'] = df['FTE_20'] - df['FTE_18']

y = df['delta_fte']
X = sm.add_constant(df['Salary_18_k'])
model = sm.OLS(y, X).fit(cov_type='HC3')

print("TABLE 1: Impact of Pandemic on FTE (First-Differences)")
print("-" * 60)
summary_table = model.summary2().tables[1]
print(summary_table[['Coef.', 'Std.Err.', 'P>|z|']])
print(f"\nN: {int(model.nobs)}")
print(f"R-squared: {model.rsquared:.3f}")
print("-" * 60)

TABLE 1: Impact of Pandemic on FTE (First-Differences)
------------------------------------------------------------
                Coef.  Std.Err.     P>|z|
const       -1.739332  5.379118  0.746431
Salary_18_k -0.045905  0.086835  0.597047

N: 21
R-squared: 0.016
------------------------------------------------------------


## 3. Interpretation of Results

### Interpretation of Main Coefficients:
1. **Constant ($\beta_0 = -1.74$):** This represents the average pandemic shock for a theoretical study area with a zero salary. However, more realistically, it highlights that even after controlling for baseline pay, employment rates dropped universally by roughly 1.7 to 4.5 percentage points across most fields.
2. **Baseline Salary ($\beta_1 = -0.046$):** 
   - **Direction:** Negative. This suggests that for every $1,000 increase in baseline salary, the employment rate dropped by an *additional* 0.046 percentage points.
   - **Magnitude:** The effect is very small. A $10,000 difference in starting salary only predicts a 0.46 percentage point difference in employment change.
   - **Significance:** With a p-value of 0.597, this result is **not statistically significant**. This implies that the pandemic's impact on employment was 'blind' to the prestige or pay grade of the field—it was a universal labor market shock.

## 4. Threats to Causal Ambition

1. **Omitted Variable Bias (OVB) - Remote-Workability:** 
   - **Threat:** High-salary sectors (e.g., Finance, Tech) often allow for remote work, while lower-salary sectors (e.g., Tourism, Hospitality) require physical presence. 
   - **Sign:** Remote-workability is positively correlated with salary ($+$) and positively correlated with employment resilience ($+$). 
   - **Direction of Bias:** By omitting 'Remote-Workability', our salary coefficient likely suffers from **upward bias** (it looks more protective than it actually is). Since our result was already non-significant, this bias suggests that high-salary fields might have actually fared even worse if they hadn't been able to work remotely.

2. **Selection Bias (Endogenous Education):**
   - Graduates in 2020 who saw the poor labor market might have **selected** into further study (Honours/Masters) rather than looking for full-time work. This means the 2020 survey sample might only include the most 'employable' graduates who stayed in the labor force, potentially understating the true severity of the treatment effect.